# Evaluation Sampling + REsearcher Annotation + Fleiss' Kappa

In [1]:
import os                                                                                                 
import numpy as np                                                                                        
import pandas as pd                                                                                       
from sklearn.model_selection import train_test_split                                                      
from statsmodels.stats.inter_rater import fleiss_kappa                                                    
                                                                                                            
# Enforce strict reproducibility                                                                          
SEED = 42                                                                                                 
np.random.seed(SEED)


In [2]:
# =============================================================================                           
# Sub-step 4.2.1: Evaluation Sampling                                                                     
# Jointly stratify by TargetAspect and proxy star rating buckets:                                         
# - 1 to 2 stars: Negative proxy                                                                          
# - 3 stars: Neutral proxy                                                                                
# - 4 to 5 stars: Positive proxy                                                                          
# Sample 400 from Validation and 400 from Test (800 instances total).                                     
# =============================================================================                           
                                                                                                            
def assign_rating_bucket(rating):                                                                         
    r = int(rating)                                                                                       
    if r <= 2:                                                                                            
        return '1-2_stars'                                                                                
    elif r == 3:                                                                                          
        return '3_stars'                                                                                  
    return '4-5_stars'                                                                                    
                                                                                                            
def draw_stratified_evaluation_sample(input_csv, sample_size=400, seed=SEED):                             
    df = pd.read_csv(input_csv)                                                                           
                                                                                                            
    # Create joint stratification key                                                                     
    df['rating_bucket'] = df['rating'].apply(assign_rating_bucket)                                        
    df['stratum'] = df['TargetAspect'].astype(str) + " | " + df['rating_bucket']                          
                                                                                                            
    # Exact stratified draw                                                                               
    sample_df, _ = train_test_split(                                                                      
        df,                                                                                               
        train_size=sample_size,                                                                           
        stratify=df['stratum'],                                                                           
        random_state=seed,                                                                                
        shuffle=True                                                                                      
    )                                                                                                     
                                                                                                            
    # Prepare annotation columns for 3 independent raters                                                 
    base_cols = ['place_name', 'rating', 'shop_type', 'sentence', 'TargetAspect']                         
    export_df = sample_df[base_cols].copy().reset_index(drop=True)                                        
    export_df['annotator_1'] = np.nan                                                                     
    export_df['annotator_2'] = np.nan                                                                     
    export_df['annotator_3'] = np.nan                                                                     
                                                                                                            
    return export_df                                                                                      
                                                                                                            
# Draw samples                                                                                            
val_gold_unannotated = draw_stratified_evaluation_sample('Validation_Set_15.csv', sample_size=400)        
test_gold_unannotated = draw_stratified_evaluation_sample('Test_Set_15.csv', sample_size=400)             
                                                                                                            
# Save unannotated templates for the 3 human raters                                                       
val_gold_unannotated.to_csv('Gold_Validation_Set_400_Unannotated.csv', index=False)                       
test_gold_unannotated.to_csv('Gold_Test_Set_400_Unannotated.csv', index=False)                            
                                                                                                            
print(f"Generated Gold_Validation_Set_400_Unannotated.csv: {val_gold_unannotated.shape}")                 
print(f"Generated Gold_Test_Set_400_Unannotated.csv: {test_gold_unannotated.shape}")

Generated Gold_Validation_Set_400_Unannotated.csv: (400, 8)
Generated Gold_Test_Set_400_Unannotated.csv: (400, 8)


In [3]:
# Verify distribution across aspects and proxy buckets                                                    
def display_sampling_strata(df, name):                                                                    
    print(f"\n=== {name} Stratified Distribution (n={len(df)}) ===")                                      
    df_temp = df.copy()                                                                                   
    df_temp['rating_bucket'] = df_temp['rating'].apply(assign_rating_bucket)                              
    ct = pd.crosstab(df_temp['TargetAspect'], df_temp['rating_bucket'], margins=True,                     
margins_name='Total')                                                                                       
    print(ct.to_markdown())                                                                               
                                                                                                            
display_sampling_strata(val_gold_unannotated, "Validation Sample")                                        
display_sampling_strata(test_gold_unannotated, "Test Sample")


=== Validation Sample Stratified Distribution (n=400) ===
| TargetAspect                       |   1-2_stars |   3_stars |   4-5_stars |   Total |
|:-----------------------------------|------------:|----------:|------------:|--------:|
| Ambiance and Atmosphere            |           3 |         3 |          47 |      53 |
| Customer Service                   |          15 |         6 |          59 |      80 |
| Facilities and Amenities           |           4 |         3 |          26 |      33 |
| Price and Value                    |           3 |         3 |          15 |      21 |
| Product Quality                    |          21 |        14 |         157 |     192 |
| Store Operations and Accessibility |           3 |         2 |          16 |      21 |
| Total                              |          49 |        31 |         320 |     400 |

=== Test Sample Stratified Distribution (n=400) ===
| TargetAspect                       |   1-2_stars |   3_stars |   4-5_stars |   Total 

# Fleiss Kappa

In [4]:
CATEGORIES = ['POSITIVE', 'NEUTRAL', 'NEGATIVE']                                                          
                                                                                                            
def clean_sentiment(label):                                                                               
    s = str(label).strip().upper()                                                                        
    if 'POS' in s:                                                                                        
        return 'POSITIVE'                                                                                 
    elif 'NEG' in s:                                                                                      
        return 'NEGATIVE'                                                                                 
    elif 'NEU' in s:                                                                                      
        return 'NEUTRAL'                                                                                  
    return s                                                                                              
                                                                                                            
def process_inter_rater_reliability(partition_name, file1, file2, file3, output_csv):                     
    # 1. Load the 3 independent annotator files                                                           
    df1 = pd.read_csv(file1)                                                                              
    df2 = pd.read_csv(file2)                                                                              
    df3 = pd.read_csv(file3)                                                                              
                                                                                                            
    # 2. Extract and standardize annotator labels (fixes whitespace & typos like 'Postive')               
    df = df1[['place_name', 'rating', 'shop_type', 'sentence', 'TargetAspect']].copy()                    
    df['annotator_1'] = df1['annotator'].apply(clean_sentiment)                                           
    df['annotator_2'] = df2['annotator'].apply(clean_sentiment)                                           
    df['annotator_3'] = df3['annotator'].apply(clean_sentiment)                                           
                                                                                                            
    # Validate integrity                                                                                  
    rater_cols = ['annotator_1', 'annotator_2', 'annotator_3']                                            
    for col in rater_cols:                                                                                
        invalid = ~df[col].isin(CATEGORIES)                                                               
        assert not invalid.any(), f"Found invalid labels in {col}: {df.loc[invalid, col].unique()}"       
                                                                                                            
                                   
    # N = 400 instances, k = 3 categories (POSITIVE, NEUTRAL, NEGATIVE)                                   
    kappa_matrix = np.zeros((len(df), len(CATEGORIES)), dtype=int)                                        
    for i, cat in enumerate(CATEGORIES):                                                                  
        kappa_matrix[:, i] = (df[rater_cols] == cat).sum(axis=1)                                          
                                                                                                            
    kappa_score = fleiss_kappa(kappa_matrix, method='fleiss')                                             
                                                                                                            
                                            
    if kappa_score > 0.80:                                                                                
        interpretation = "Almost Perfect Agreement"                                                       
    elif kappa_score > 0.60:                                                                              
        interpretation = "Substantial Agreement"                                                          
    elif kappa_score > 0.40:                                                                              
        interpretation = "Moderate Agreement"                                                             
    else:                                                                                                 
        interpretation = "Fair / Poor Agreement"                                                          
                                                                                                            
                                                    
    def get_consensus(row):                                                                               
        votes = [row['annotator_1'], row['annotator_2'], row['annotator_3']]                              
        val_counts = pd.Series(votes).value_counts()                                                      
        if val_counts.iloc[0] >= 2:                                                                       
            return pd.Series([val_counts.index[0], False, f"{val_counts.iloc[0]}/3"])                     
        else:                                                                                             
            # 3-way tie fallback (1 POS, 1 NEU, 1 NEG)                                                    
            return pd.Series([row['annotator_1'], True, "1/1/1_tie"])                                     
                                                                                                            
    df[['ground_truth', 'is_tie', 'vote_breakdown']] = df.apply(get_consensus, axis=1)                    
                                                                                                            
                                                                      
    print(f"\n=======================================================")                                   
    print(f" {partition_name} Set (n={len(df)}) Inter-Rater Reliability")                                 
    print(f"=======================================================")                                     
    print(f"Fleiss' Kappa (κ): {kappa_score:.4f} ({interpretation})")                                     
    print(f"Tied Discrepancies (3-way split): {df['is_tie'].sum()}")                                      
    print("\nDefinitive Ground Truth Sentiment Breakdown:")                                               
    print(df['ground_truth'].value_counts().to_string())                                                  
                                                                                                            
                                                          
    df.to_csv(output_csv, index=False)                                                                    
    print(f"\nSuccessfully exported definitive answer key to: {output_csv}")                              
    return df                                                                                             
                                                                                                            
                                                                            
val_final = process_inter_rater_reliability(                                                              
    partition_name="Validation",                                                                          
    file1="gold_Validation_Set_400_annotated1_completed.csv",                                             
    file2="gold_Validation_Set_400_annotated2_completed.csv",                                             
    file3="gold_Validation_Set_400_annotated3_completed.csv",                                             
    output_csv="Gold_Validation_Set_400_Final.csv"                                                        
)                                                                                                         
                                                                                                            
# Run on Test Set (400)                                                                                   
test_final = process_inter_rater_reliability(                                                             
    partition_name="Test",                                                                                
    file1="gold_test_set_400_annotated1_completed.csv",                                                   
    file2="gold_test_set_400_annotated2_completed.csv",                                                   
    file3="gold_test_set_400_annotated3_completed.csv",                                                   
    output_csv="Gold_Test_Set_400_Final.csv"                                                              
)       


 Validation Set (n=400) Inter-Rater Reliability
Fleiss' Kappa (κ): 1.0000 (Almost Perfect Agreement)
Tied Discrepancies (3-way split): 0

Definitive Ground Truth Sentiment Breakdown:
ground_truth
POSITIVE    278
NEGATIVE    100
NEUTRAL      22

Successfully exported definitive answer key to: Gold_Validation_Set_400_Final.csv

 Test Set (n=400) Inter-Rater Reliability
Fleiss' Kappa (κ): 1.0000 (Almost Perfect Agreement)
Tied Discrepancies (3-way split): 0

Definitive Ground Truth Sentiment Breakdown:
ground_truth
POSITIVE    294
NEGATIVE     73
NEUTRAL      33

Successfully exported definitive answer key to: Gold_Test_Set_400_Final.csv
